# Manifold-Constrained Hyper-Connections
## Part 2: Do multiple residual streams help — and should stream selection be learned?

Part 1 compared optimizers (AdamW vs Muon). Here we hold the optimizer fixed (**Muon**) and vary the **connection scheme** between sublayers.

**mHC** keeps `n` parallel residual streams instead of one. Two independent learnable pieces control the flow:
- **Mixing** — Sinkhorn-normalized doubly-stochastic matrices `A` (before the sublayer) and `B` (after) mix information across streams. Initialized to ~identity, so every run starts at the standard residual baseline.
- **Selection** — how the sublayer *reads* its input from the streams and *writes* its output back.

We answer two questions:
1. **mHC vs no mHC** — does running parallel streams help at all?
2. **Static vs learnable selection** — the original code hard-codes stream 0 as the read/write slot. Does *learning* the selection do better?

To isolate selection from mixing, we run a **2×2 (selection × mixing)** plus the no-mHC baseline:

| Run | streams | mixing (A/B) | selection |
|---|---|---|---|
| `baseline_no_mhc` | 1 | — | standard residual |
| `mhc_static` | 4 | yes | fixed slot 0 |
| `mhc_learnable` | 4 | yes | learned read/write vectors |
| `mhc_static_nomix` | 4 | **no** | fixed slot 0 |
| `mhc_selection_only` | 4 | **no** | learned read/write vectors |
| `mhc_per_token` | 4 | yes | data-dependent (per-token gate) |

> **Why the 2×2 matters.** With mixing on, "static" isn't truly static: the read is `(A·S)[0] = Σⱼ A[0,j]·Sⱼ`, so `A`/`B` already make the effective read/write partly learnable — comparing `mhc_static` vs `mhc_learnable` conflates *selection* with *mixing*. The clean test of selection alone is **mixing off**: `mhc_static_nomix` vs `mhc_selection_only`, where the only difference is fixed slot-0 vs learned read/write.

Logs to W&B project `mhc-nanogpt-blog`, group `mhc-streams`. (This is a **new** notebook — Part 1 stays untouched.)

In [ ]:
!pip install -q git+https://github.com/ashishjv1/mHC.git
!pip install -q wandb

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
import wandb

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

from src.muon import Muon
from src.model import GPT
from configs.train_config import TrainConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass
wandb.login()

---
## Data: WikiText-2

Same dataset and tokenizer as Part 1, so the loss numbers are directly comparable.

In [ ]:
import tiktoken
from datasets import load_dataset

# Salesforce/wikitext is the Parquet copy (no loading script), so it works on
# recent `datasets` versions that removed trust_remote_code. Same configs/fields.
print('Downloading WikiText-2...')
ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
enc = tiktoken.get_encoding('gpt2')

all_tokens = []
for row in ds:
    text = row['text'].strip()
    if text:
        all_tokens.extend(enc.encode_ordinary(text))
        all_tokens.append(enc.eot_token)
all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print(f'WikiText-2: {len(all_tokens):,} tokens')


def get_batch(tokens, batch_size, ctx_len):
    ix = torch.randint(len(tokens) - ctx_len - 1, (batch_size,))
    x = torch.stack([tokens[i:i+ctx_len] for i in ix]).to(device)
    y = torch.stack([tokens[i+1:i+1+ctx_len] for i in ix]).to(device)
    return x, y

---
## Training

Every run shares the same model size, data, optimizer (Muon), seed and step budget. **Only the connection scheme changes**, so loss differences are attributable to mHC / selection, not to anything else.

In [ ]:
def train_run(run_name, overrides, tokens):
    """Train a GPT on WikiText-2 with Muon. `overrides` sets the mHC scheme."""
    base = dict(
        n_layers=6, d_model=512, n_heads=8, d_ff=2048,
        vocab_size=50304, context_len=256,
        dropout=0.0, bias=False, use_mhc=False, compile=False,
        optimizer='muon',
        lr=3e-4, min_lr=3e-5,
        muon_lr=0.02, muon_min_lr=0.002, muon_momentum=0.95, muon_ns_steps=5,
        weight_decay=0.1, batch_size=32, grad_accum_steps=1,
        max_steps=1000, warmup_steps=100,
    )
    cfg = TrainConfig(**{**base, **overrides})

    torch.manual_seed(42)
    wandb.init(project='mhc-nanogpt-blog', name=run_name, group='mhc-streams',
               tags=['blog-part2', 'mhc'],
               config={'use_mhc': cfg.use_mhc, 'n_streams': cfg.n_streams,
                       'hc_selection': cfg.hc_selection, 'hc_mix': cfg.hc_mix,
                       'd_model': cfg.d_model, 'n_layers': cfg.n_layers,
                       'max_steps': cfg.max_steps})

    model = GPT(cfg).to(device)
    print(f'{run_name}: {model.count_parameters():,} params '
          f'(mhc={cfg.use_mhc}, selection={cfg.hc_selection}, mix={cfg.hc_mix})')

    # Muon for 2D weight matrices, AdamW for everything else (incl. all hc params)
    muon_params, adamw_decay, adamw_nodecay = model.get_param_groups()
    muon_opt = Muon([{'params': muon_params}], lr=cfg.muon_lr,
                    momentum=cfg.muon_momentum, ns_steps=cfg.muon_ns_steps)
    opt = torch.optim.AdamW([
        {'params': adamw_decay, 'weight_decay': cfg.weight_decay},
        {'params': adamw_nodecay, 'weight_decay': 0.0},
    ], lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    def lr_at(step, peak, floor):
        if step < cfg.warmup_steps:
            return peak * (step + 1) / cfg.warmup_steps
        progress = (step - cfg.warmup_steps) / (cfg.max_steps - cfg.warmup_steps)
        return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * progress))

    losses = []
    for step in range(cfg.max_steps):
        lr = lr_at(step, cfg.lr, cfg.min_lr)
        mlr = lr_at(step, cfg.muon_lr, cfg.muon_min_lr)
        for g in opt.param_groups:
            g['lr'] = lr
        for g in muon_opt.param_groups:
            g['lr'] = mlr

        x, y = get_batch(tokens, cfg.batch_size, cfg.context_len)
        model.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        muon_opt.step()
        opt.step()

        losses.append(loss.item())
        wandb.log({'train/loss': loss.item(), 'train/lr': lr}, step=step)
        if step % 200 == 0:
            print(f'  step {step:>4d} | loss {loss.item():.4f}')

    wandb.run.summary['final_loss'] = float(np.mean(losses[-50:]))
    wandb.finish()
    return losses

In [ ]:
EXPERIMENTS = [
    ('baseline_no_mhc',    dict(use_mhc=False)),
    ('mhc_static',         dict(use_mhc=True, n_streams=4, hc_selection='static',    hc_mix=True)),
    ('mhc_learnable',      dict(use_mhc=True, n_streams=4, hc_selection='learnable', hc_mix=True)),
    ('mhc_static_nomix',   dict(use_mhc=True, n_streams=4, hc_selection='static',    hc_mix=False)),
    ('mhc_selection_only', dict(use_mhc=True, n_streams=4, hc_selection='learnable', hc_mix=False)),
    ('mhc_per_token',      dict(use_mhc=True, n_streams=4, hc_selection='per_token', hc_mix=True)),
]

results = {}
for name, overrides in EXPERIMENTS:
    print(f'\n=== {name} ===')
    results[name] = train_run(name, overrides, all_tokens)

---
## Results

In [ ]:
w = 20
colors = {
    'baseline_no_mhc': '#7f7f7f',
    'mhc_static': '#1f77b4',
    'mhc_learnable': '#ff7f0e',
    'mhc_static_nomix': '#9467bd',
    'mhc_selection_only': '#2ca02c',
    'mhc_per_token': '#d62728',
}

fig, ax = plt.subplots(figsize=(11, 6))
for name, losses in results.items():
    smooth = np.convolve(losses, np.ones(w)/w, mode='valid')
    ax.plot(smooth, color=colors.get(name), linewidth=1.6, alpha=0.9, label=name)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (smoothed)')
ax.set_title('mHC stream-selection ablation on WikiText-2 (6L / 512d, Muon, 1000 steps)')
ax.legend()
plt.tight_layout()
plt.show()

print('Final loss (mean of last 50 steps):')
base = np.mean(results['baseline_no_mhc'][-50:])
for name, losses in results.items():
    fl = np.mean(losses[-50:])
    delta = fl - base
    print(f'  {name:20s} {fl:.4f}   ({delta:+.4f} vs baseline)')

---
## How to read this

- **Does mHC help?** — compare `baseline_no_mhc` against the `mhc_*` runs. Since every mHC run starts at the residual baseline (≈identity mixing, slot-0 selection), a consistent gap is attributable to mHC, not to a different starting point.
- **Static vs learnable selection (the clean test)** — `mhc_static_nomix` vs `mhc_selection_only`. Both have mixing **off**, so the *only* difference is fixed slot-0 vs learned read/write. This is the controlled answer to "should selection be learned?"
- **Static vs learnable, with mixing on** — `mhc_static` vs `mhc_learnable` / `mhc_per_token`. Useful, but remember mixing makes even "static" partly learnable, so read this together with the mixing-off pair.
- **Role of mixing** — `mhc_static` vs `mhc_static_nomix` (and `mhc_learnable` vs `mhc_selection_only`) isolates what the Sinkhorn `A`/`B` matrices add on their own.

The 2×2 view (selection × mixing):

| | mixing **on** | mixing **off** |
|---|---|---|
| **static** | `mhc_static` | `mhc_static_nomix` |
| **learnable** | `mhc_learnable` | `mhc_selection_only` |

Caveat: 1000 steps on WikiText-2 is a fast directional probe, not a final verdict. If a variant looks promising, re-run with more seeds / steps before drawing conclusions.

All runs are on W&B under `mhc-nanogpt-blog`, group `mhc-streams`.